In [ ]:
!pip install --upgrade --index-url https://pypi.ngc.nvidia.com nvidia-tensorrt
!sudo apt-get install -y tensorrt

Looking in indexes: https://pypi.ngc.nvidia.com
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╺━ 324.0/340.9 MB 14.0 MB/s eta 0:00:02

In [ ]:
import tensorrt
print(tensorrt.__version__)
assert tensorrt.Builder(tensorrt.Logger())

In [ ]:
!sudo apt-get install python3-libnvinfer-dev -y
!python3 -m pip install numpy onnx
!sudo apt-get install onnx-graphsurgeon -y
#--------------------------------------------------
#upgrade pytorch to 1.12
#!pip install torch==1.12.1+cu116 torchvision==0.13.1+cu116 torchaudio==0.12.1 --extra-index-url https://download.pytorch.org/whl/cu116

In [ ]:
!dpkg-query -W tensorrt
!dpkg -l | grep nvinfer

In [ ]:
!pip install --upgrade pip
!pip install --upgrade sphinx-glpi-theme
!pip install --no-cache-dir --index-url https://pypi.nvidia.com pytorch-quantization
!pip install wget
!pip3 install -U pip && pip3 install onnxsim
!pip install onnxruntime

In [ ]:
# !pip install git+https://github.com/huggingface/accelerate

In [ ]:
# !pip show accelerate

In [ ]:
!ls /opt/conda/lib/python3.7/site-packages/onnxruntime/tools/

In [ ]:
!pip install polygraphy


In [ ]:
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"


In [ ]:
torch.cuda.empty_cache()


In [ ]:
import torch
import torch.nn as nn
import torch.backends.cudnn as cudnn
import torch.nn.functional as F
import torch.optim as optim
import torch.utils.data as data
import torchvision
import torchvision.transforms as transforms
from torchvision import models, datasets
from torch.utils.tensorboard import SummaryWriter
import onnxruntime
import onnxsim


import pytorch_quantization
from pytorch_quantization import nn as quant_nn
from pytorch_quantization import quant_modules
from pytorch_quantization import calib
from pytorch_quantization.tensor_quant import QuantDescriptor
from pytorch_quantization.nn.modules.tensor_quantizer import TensorQuantizer
from pytorch_quantization.nn.modules import quant_pooling
from tqdm import tqdm

print(pytorch_quantization.__version__)
# print(torch_tensorrt.__version__)
print(onnxruntime.__version__)
print(onnxsim.__version__)
#print(torchaudio.__version__)

import os
import sys
import warnings
import time
import numpy as np
import wget
import tarfile
import shutil
warnings.simplefilter('ignore')
import collections

# Accelerate parts
from accelerate import Accelerator, notebook_launcher # main interface, distributed launcher
from accelerate.utils import set_seed # reproducability across devices

In [ ]:
# Initialize the Accelerator
# accelerator = Accelerator()

# Set seed for reproducibility
set_seed(42)


In [ ]:
from PIL import Image
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2
import os

class CityscapesDataset(Dataset):
    def __init__(self, root_dir, split='train', resize_shape=(256, 512), is_train=True):
        super().__init__()
        self.inputs, self.targets = [], []
        self.resize_shape = resize_shape
        self.ignore_index = 255
        self.valid_classes = [7, 8, 11, 12, 13, 17, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 31, 32, 33]
        self.class_map = {k: v for v, k in enumerate(self.valid_classes)}
        self.is_train = is_train

        img_dir = os.path.join(root_dir, 'leftImg8bit', split)
        mask_dir = os.path.join(root_dir, 'gtFine', split)

        for root, _, filenames in os.walk(img_dir):
            for filename in filenames:
                if filename.endswith('.png'):
                    filename_base = '_'.join(filename.split('_')[:-1])
                    input_path = os.path.join(root, filename_base + '_leftImg8bit.png')
                    target_path = os.path.join(mask_dir, os.path.basename(root), filename_base + '_gtFine_labelIds.png')

                    if os.path.exists(input_path) and os.path.exists(target_path):
                        self.inputs.append(input_path)
                        self.targets.append(target_path)

        print(f"Found {len(self.inputs)} images and {len(self.targets)} targets for split '{split}'")

        # Define Albumentations transformation pipeline
        if self.is_train:
            self.transform = A.Compose([
                A.Resize(self.resize_shape[0], self.resize_shape[1]),  # Ensure consistent size
                A.HorizontalFlip(p=0.5),
                A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1, p=0.5),
                A.GaussianBlur(blur_limit=(3, 7), p=0.2),  # Add some blur
                A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
                ToTensorV2()
            ])
        else:
            self.transform = A.Compose([
                A.Resize(self.resize_shape[0], self.resize_shape[1]),
                A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
                ToTensorV2()
            ])

    def __len__(self):
        return len(self.inputs)

    def __getitem__(self, i):
        input_img = np.array(Image.open(self.inputs[i]).convert('RGB'))
        target_img = np.array(Image.open(self.targets[i]))

        # Apply transformations to both image and mask
        transformed = self.transform(image=input_img, mask=target_img)
        input_img = transformed['image']
        target_tensor = transformed['mask']

        # Remap target labels
        remapped_target = torch.full_like(target_tensor, self.ignore_index)
        for cls, mapped in self.class_map.items():
            remapped_target[target_tensor == cls] = mapped

        # Debugging output for shapes
        assert input_img.shape[1:] == remapped_target.shape, f"Shape mismatch: input {input_img.shape[1:]}, target {remapped_target.shape}"

        return input_img, remapped_target


# DataLoader setup
root_dir = '/kaggle/input/cityscapes/Cityscape'
resize_shape = (512, 1024)

train_dataset = CityscapesDataset(root_dir=root_dir, split='train', resize_shape=resize_shape, is_train=True)
val_dataset = CityscapesDataset(root_dir=root_dir, split='val', resize_shape=resize_shape, is_train=False)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=4, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False, num_workers=4, pin_memory=True)
inference_loader = DataLoader(val_dataset, batch_size=1, shuffle=False, num_workers=4, pin_memory=True)



In [ ]:
import matplotlib.pyplot as plt
import torch

# Decode function to convert mask to RGB
def decode_segmap(mask, n_classes=20, label_colours=None):
    if label_colours is None:
        label_colours = [
            (102, 102, 156), (70, 70, 70), (244, 35, 232), (128, 64, 128),
            (190, 153, 153), (153, 153, 153), (250, 170, 30), (220, 220, 0),
            (107, 142, 35), (152, 251, 152), (0, 130, 180), (220, 20, 60),
            (255, 0, 0), (0, 0, 142), (0, 0, 70), (0, 60, 100),
            (0, 80, 100), (0, 0, 230), (119, 11, 32), (120, 170, 250)
        ]

    # Convert to numpy array if needed
    if isinstance(mask, torch.Tensor):
        mask = mask.cpu().numpy()

    # Initialize empty RGB channels
    r, g, b = np.zeros_like(mask, dtype=np.uint8), np.zeros_like(mask, dtype=np.uint8), np.zeros_like(mask, dtype=np.uint8)

    # Apply color mapping
    for l in range(n_classes):
        r[mask == l] = label_colours[l][0]
        g[mask == l] = label_colours[l][1]
        b[mask == l] = label_colours[l][2]

    # Stack to form an RGB image
    rgb = np.stack([r, g, b], axis=2)  # Result shape: [height, width, 3]
    return rgb

# Visualization function for input, ground truth, and predicted segmentation
def visualize_segmentation(input_image, decoded_target, decoded_prediction):
    # Display the images
    fig, axs = plt.subplots(1, 3, figsize=(18, 6))
    axs[0].imshow(input_image.permute(1, 2, 0).cpu().numpy())  # Convert input tensor to numpy for display
    axs[0].set_title("Input Image")
    axs[0].axis('off')

    axs[1].imshow(decoded_target)
    axs[1].set_title("Ground Truth Mask")
    axs[1].axis('off')

    axs[2].imshow(decoded_prediction)
    axs[2].set_title("Predicted Mask")
    axs[2].axis('off')

    plt.show()

# Function to perform inference and visualize predictions
def visualize_predictions(model, data_loader, device, n_classes=20):
    model.eval()  # Set the model to evaluation mode
    with torch.no_grad():
        for input_img, target_mask in data_loader:
            # Move data to the appropriate device
            input_img, target_mask = input_img.to(device), target_mask.to(device)

            # Perform inference
            output = model(input_img)['out']  # Assuming DeepLabV3 output format
            _, predicted_mask = torch.max(output, 1)  # Get predicted class indices

            # Convert single image from batch
            input_image = input_img[0]  # Select the first image from the batch
            target_mask = target_mask[0]  # Select the corresponding ground truth mask
            predicted_mask = predicted_mask[0]  # Select the corresponding predicted mask

            # Decode ground truth and prediction masks only once
            decoded_target = decode_segmap(target_mask, n_classes=n_classes)
            decoded_prediction = decode_segmap(predicted_mask, n_classes=n_classes)

            # Visualize the segmentation results
            visualize_segmentation(input_image, decoded_target, decoded_prediction)
            break  # Visualize only one batch for now

# Usage example:
# Assuming `trained_model` is the pre-trained DeepLabV3 model and `device` is the device ('cpu' or 'cuda')
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
trained_model = model.to(device)  # Ensure model is on the correct device

# Call the visualization function with the model and data loader
visualize_predictions(model, inference_loader, device, n_classes=19)


In [ ]:
# Call the visualization function with the model and data loader
visualize_predictions(q_model, inference_loader, device, n_classes=19)


In [ ]:
from torchvision.models.segmentation import lraspp_mobilenet_v3_large



def replace_relu_with_silu(module):
    for name, child in module.named_children():
        if isinstance(child, nn.ReLU):
            setattr(module, name, nn.SiLU())
        else:
            replace_relu_with_silu(child)

model = lraspp_mobilenet_v3_large(pretrained=False) # Load the model pre-trained on COCO
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

num_classes = 19  # Cityscapes has 19 valid classes
# Modify low_classifier and high_classifier for 19 classes
# Assuming `model` is your LRASPP-based model
# replace_relu_with_silu(model)
model.classifier.low_classifier = torch.nn.Conv2d(40, 19, kernel_size=1, stride=1)
model.classifier.high_classifier = torch.nn.Conv2d(128, 19, kernel_size=1, stride=1)

# Wrap the model in DataParallel and move to main device
if torch.cuda.device_count() > 1:
    print("Using", torch.cuda.device_count(), "GPUs!")
    # model = nn.DataParallel(model)  # Wrap model in DataParallel

# Move the model to the GPU if available
# model = model.to(accelerator.device)
model = model.to(device)



In [ ]:
checkpoint_path = "/kaggle/input/lsarpp_mobilenetv3_largev5/pytorch/default/1/best_model_iou_v5.pth"

# ✅ Load checkpoint if available
def load_checkpoint(model, optimizer, checkpoint_path, device):
    if os.path.exists(checkpoint_path):
        checkpoint = torch.load(checkpoint_path, map_location=device)
        model.load_state_dict(checkpoint['model_state_dict'])
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        start_epoch = checkpoint['epoch']
        best_val_iou = checkpoint['val_iou']
        print(f"✅ Checkpoint loaded: Resuming from Epoch {start_epoch} with Best IoU {best_val_iou:.4f}")
        return model, optimizer, start_epoch, best_val_iou
    else:
        print("⚠ No checkpoint found. Starting training from scratch.")
        return model, optimizer, 0, 0.0

model, optimizer, start_epoch, best_val_iou = load_checkpoint(model, optimizer, checkpoint_path, device)



In [ ]:
import torch
from collections import OrderedDict  # Import OrderedDict


# Load the full checkpoint dictionary
checkpoint = torch.load("/kaggle/input/lsarpp_mobilenetv3_largev5/pytorch/default/1/best_model_iou_v5.pth")

# Remove the "module." prefix from the keys in state_dict if necessary
state_dict = checkpoint['model_state_dict']
new_state_dict = OrderedDict()
for k, v in state_dict.items():
    name = k.replace("module.", "")  # Remove `module.` prefix
    new_state_dict[name] = v

# Load the model state
model.load_state_dict(new_state_dict)

# Load the optimizer state (optional, only if you want to continue training)
optimizer.load_state_dict(checkpoint['optimizer_state_dict'])

# Retrieve additional information
epoch = checkpoint['epoch']
# val_accuracy = checkpoint['val_accuracy']

# print(f"Model loaded from epoch {epoch} with validation accuracy {val_accuracy* 100:.2f}%")

# Move model to the appropriate device if needed
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# Set model to evaluation mode if you're using it for inference
model.eval()


In [ ]:
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.optim.lr_scheduler import OneCycleLR

# Loss function with ignore_index for unlabeled pixels
######Exp1############
# optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
# criterion = nn.CrossEntropyLoss(ignore_index=255)
# scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.1, patience=3, verbose=True)
#-----------------Exp2-----------------------------------
criterion = nn.CrossEntropyLoss(ignore_index=255)
optimizer = optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
# scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10, eta_min=1e-6)

# scheduler = OneCycleLR(optimizer, max_lr=3e-4, epochs=30, steps_per_epoch=len(train_loader), pct_start=0.3, anneal_strategy='cos', div_factor=10)
scheduler  = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.3, patience=5, threshold=0.001, verbose=True)

In [ ]:
from sklearn.metrics import confusion_matrix
from torch.nn import CrossEntropyLoss

# Helper functions for accuracy and IoU
def calculate_accuracy(outputs, targets):
    _, preds = torch.max(outputs, 1)
    correct = (preds == targets).float().sum()
    return correct / targets.numel()


def calculate_mean_iou(preds, targets, n_classes):
    preds = preds.cpu().numpy().flatten()
    targets = targets.cpu().numpy().flatten()
    mask = targets != 255  # Ignore unlabeled pixels
    preds = preds[mask]
    targets = targets[mask]
    if len(preds) == 0:
        return 0.0
    cm = confusion_matrix(targets, preds, labels=np.arange(n_classes))
    intersection = np.diag(cm)
    union = cm.sum(axis=1) + cm.sum(axis=0) - intersection
    iou = intersection / (union + 1e-10)
    return np.nanmean(iou)


# Training loop
def train_one_epoch(model, train_loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    running_accuracy = 0.0

    for images, masks in train_loader:
        images, masks = images.to(device), masks.to(device).long()
        #mask[mask == 255] = 0
        optimizer.zero_grad()
        outputs = model(images)  # LRASPP has only 'out'
        main_output = outputs['out']
        loss = criterion(main_output, masks)
        loss.backward()
        optimizer.step()
        accuracy = calculate_accuracy(main_output, masks)
        running_loss += loss.item()
        running_accuracy += accuracy.item()

    epoch_loss = running_loss / len(train_loader)
    epoch_accuracy = running_accuracy / len(train_loader)
    return epoch_loss, epoch_accuracy


# Validation loop with mean IoU calculation
def evaluate_model_with_metrics(model, val_loader, criterion, device, n_classes):
    model.eval()
    val_loss, val_accuracy, val_iou = 0.0, 0.0, 0.0
    with torch.no_grad():
        for images, masks in val_loader:
            images, masks = images.to(device), masks.to(device).long()
            outputs = model(images)
            main_output = outputs['out']
            loss = criterion(main_output, masks)
            accuracy = calculate_accuracy(main_output, masks)
            _, preds = torch.max(main_output, 1)
            iou = calculate_mean_iou(preds, masks, n_classes)
            val_loss += loss.item()
            val_accuracy += accuracy.item()
            val_iou += iou
            torch.cuda.empty_cache()
    avg_val_loss = val_loss / len(val_loader)
    avg_val_accuracy = val_accuracy / len(val_loader)
    avg_iou = val_iou / len(val_loader)
    return avg_val_loss, avg_val_accuracy, avg_iou


# EarlyStopping class for training monitoring
class EarlyStopping:
    def __init__(self, patience=5, min_delta=0.001):
        self.patience = patience
        self.min_delta = min_delta
        self.best_score = None
        self.counter = 0
        self.early_stop = False

    def __call__(self, current_score):
        if self.best_score is None:
            self.best_score = current_score
        elif current_score < self.best_score + self.min_delta:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_score = current_score
            self.counter = 0  # Reset patience if an improvement is found



# Function to save the model
def save_checkpoint(model, optimizer, epoch, val_iou, path):
    checkpoint = {
        'epoch': epoch + 1,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'val_iou': val_iou  # Save the best IoU value
    }
    torch.save(checkpoint, path)
    print(f"Checkpoint saved with IoU: {val_iou:.4f} at {path}")


In [ ]:
import os

# n_classes = 19
num_epochs = 60
patience = 10  # Stop training if no improvement for 5 epochs
early_stopping = EarlyStopping(patience=patience, min_delta=0.005)
best_val_iou = 0.0  # Track the best IoU value
log_path = '/kaggle/working/training_log-mobilenetv3_v5.txt'


# Open log file
with open(log_path, 'w') as log_file:
    # if start_epoch == 0:
    log_file.write("Epoch, Train Loss, Train Acc, Val Loss, Val Acc, Val IoU\n")

    # Initialize best IoU
    best_val_iou = 0.0  

    # Training loop with early stopping
    for epoch in range(num_epochs):
        # Train the model for one epoch
        train_loss, train_accuracy = train_one_epoch(model, train_loader, criterion, optimizer, device)
        val_loss, val_accuracy, val_iou = evaluate_model_with_metrics(model, val_loader, criterion, device, num_classes)

        # Print metrics for the current epoch
        print(f"Epoch {epoch + 1}/{num_epochs}")
        print(f"Train Loss: {train_loss:.4f}, Train Acc: {train_accuracy:.4f}")
        print(f"Validation Loss: {val_loss:.4f}, Validation Acc: {val_accuracy:.4f}, Validation IoU: {val_iou:.4f}")

        # Log metrics to file
        log_file.write(f"{epoch + 1}, {train_loss:.4f}, {train_accuracy:.4f}, {val_loss:.4f}, {val_accuracy:.4f}, {val_iou:.4f}\n")

        # Save the model checkpoint if validation IoU improves
        if val_iou > best_val_iou:
            best_val_iou = val_iou
            save_checkpoint(model, optimizer, epoch, val_iou, path='/kaggle/working/best_model_iou_v5.pth')

        # ✅ Switch to fine-tuning phase after OneCycleLR ends
        # if epoch == 40:
        #     optimizer.param_groups[0]['lr'] = 1e-4  # Reduce LR manually
        #     scheduler = fine_tune_scheduler  # Switch to ReduceLROnPlateau
        
        # Early stopping (based on IoU instead of accuracy)
        early_stopping(val_iou)
        if early_stopping.early_stop:
            print("Early stopping triggered. Stopping training.")
            break

        # ✅ Apply scheduler
        # if isinstance(scheduler, OneCycleLR):
        #     scheduler.step()
        # else:
        scheduler.step(val_iou)  # Reduce LR if IoU is not improving
print("Training complete!")



In [ ]:
val_loss, val_accuracy, val_iou = evaluate_model_with_metrics(model, val_loader, criterion, device, num_classes)
print(f"Validation Loss: {val_loss:.4f}, Validation Acc: {val_accuracy:.4f}, Validation IoU: {val_iou:.4f}")

In [ ]:
# Check if the model is wrapped in DataParallel
if isinstance(model, torch.nn.DataParallel):
    model = model.module  # Access the original model inside DataParallel


In [ ]:
import torch
import torchvision.models as models


# Dummy input tensor (assuming Cityscapes image size)
dummy_input = torch.randn(1, 3, 512, 1024).cuda()

# Export to ONNX
torch.onnx.export(
    model, dummy_input, "lsarpp_mobilenetv3_large.onnx",
    opset_version=11,   # Set compatible opset version
    input_names=["input"], output_names=["output"]
)

print("✅ Model exported successfully!")


In [ ]:
import onnx
from onnx.tools import update_model_dims

# Load ONNX model
model_path = "/kaggle/working/fixed_lsarpp_mobilenetv3_large.onnx"
onnx_model = onnx.load(model_path)

# Manually update output shape
for output in onnx_model.graph.output:
    if output.name == "output":  # Ensure we're modifying the correct output
        output.type.tensor_type.shape.dim[0].dim_value = 1
        output.type.tensor_type.shape.dim[1].dim_value = 19
        output.type.tensor_type.shape.dim[2].dim_value = 512
        output.type.tensor_type.shape.dim[3].dim_value = 1024

# Save the modified model
onnx.save(onnx_model, "fixed_output_lsarpp_mobilenetv3_large.onnx")

print("✅ ONNX output shape is now fixed!")


In [ ]:
!/usr/src/tensorrt/bin/trtexec --onnx=/kaggle/working/fixed_output_lsarpp_mobilenetv3_large.onnx --saveEngine=/kaggle/working/base_fp32.engine


In [ ]:
!/usr/src/tensorrt/bin/trtexec --onnx=/kaggle/working/fixed_output_lsarpp_mobilenetv3_large.onnx --fp16 --saveEngine=/kaggle/working/lsarpp_fp16.engine


In [ ]:
!/usr/src/tensorrt/bin/trtexec --onnx=/kaggle/working/fixed_output_lsarpp_mobilenetv3_large.onnx --int8 --calib=entropy --saveEngine=/kaggle/working/lsarpp_int8.engine


In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

# Function to compute layer sensitivities and parameter sizes
def compute_layer_sensitivity_and_size(model, loss_fn, data_loader, epsilon=1e-3, num_samples=10, device='cuda'):
    sensitivities = {}
    parameter_sizes = {}
    model.to(device).eval()

    for name, layer in model.named_modules():
        if isinstance(layer, nn.Conv2d):
            # Calculate parameter size in MB
            num_params = sum(p.numel() for p in layer.parameters())
            parameter_sizes[name] = num_params * 8 / (8 * 1024 * 1024)  # Convert to MB

            # Compute sensitivity
            total_gradient_norm = 0.0
            original_weights = [p.data.clone() for p in layer.parameters()]

            for batch_idx, (inputs, targets) in enumerate(data_loader):
                inputs = inputs.to(device).float()
                targets = targets.to(device).long()

                for _ in range(num_samples):
                    for p in layer.parameters():
                        perturbation = torch.randn_like(p, device=device) * epsilon / torch.norm(p)
                        p.data += perturbation

                    outputs = model(inputs)
                    main_output = outputs['out'] if isinstance(outputs, dict) else outputs
                    loss = loss_fn(main_output, targets)

                    model.zero_grad()
                    loss.backward()

                    gradient_norm = sum(torch.norm(p.grad, p=1).item() for p in layer.parameters())
                    total_gradient_norm += gradient_norm

                    for p, orig in zip(layer.parameters(), original_weights):
                        p.data.copy_(orig)

                if batch_idx == 0:
                    break

            sensitivities[name] = total_gradient_norm / (num_samples * (batch_idx + 1))

    return sensitivities, parameter_sizes

import matplotlib.pyplot as plt

def dp_optimize(sensitivities, parameter_sizes, bit_precisions, target_model_size):
    """
    Optimize layer bit precisions using dynamic programming and visualize results.

    Args:
        sensitivities (dict): Sensitivity scores for each layer.
        parameter_sizes (dict): Parameter sizes (in MB) for each layer.
        bit_precisions (list): List of available bit precision options.
        target_model_size (float): Target total model size (in MB).

    Returns:
        tuple: Best Pareto-optimal solution and a list of all Pareto-optimal solutions.
    """
    layers = list(sensitivities.keys())
    n_layers = len(layers)
    max_size = int(target_model_size * 1024)  # Convert MB to KB for DP indexing

    # Initialize DP table
    dp_table = [{} for _ in range(n_layers + 1)]
    dp_table[0][0] = (0, {})  # (accuracy_loss, bit_precision_config)

    # Fill DP table
    for i, layer in enumerate(layers, start=1):
        sensitivity = sensitivities[layer]
        param_size = parameter_sizes[layer] * 1024  # Convert MB to KB for DP indexing

        for prev_size, (prev_loss, prev_config) in dp_table[i - 1].items():
            for precision in bit_precisions:
                layer_size = param_size * (precision / 8.0)  # Compute size for the given precision
                new_size = prev_size + int(layer_size)

                if new_size > max_size:
                    continue

                accuracy_loss = prev_loss + sensitivity * (1 / precision)  # Quantization error inversely proportional to precision
                new_config = prev_config.copy()
                new_config[layer] = precision

                if new_size not in dp_table[i] or dp_table[i][new_size][0] > accuracy_loss:
                    dp_table[i][new_size] = (accuracy_loss, new_config)

    # Extract Pareto-optimal solutions
    pareto_solutions = []
    for model_size, (accuracy_loss, bit_config) in dp_table[n_layers].items():
        pareto_solutions.append((model_size / 1024, accuracy_loss, bit_config))  # Convert size back to MB

    pareto_solutions.sort(key=lambda x: x[1])  # Sort by accuracy loss

    # Visualization of Pareto-Frontier
    sizes = [solution[0] for solution in pareto_solutions]
    losses = [solution[1] for solution in pareto_solutions]

    plt.figure(figsize=(10, 6))
    plt.plot(sizes, losses, marker='o', label='Pareto-Frontier')
    plt.xlabel("Model Size (MB)")
    plt.ylabel("Accuracy Loss")
    plt.title("Pareto-Frontier: Accuracy Loss vs. Model Size")
    plt.grid(alpha=0.5)
    plt.legend()
    plt.tight_layout()
    plt.show()

    # Return the best solution (smallest accuracy loss) and all Pareto solutions
    best_solution = pareto_solutions[0]
    return best_solution, pareto_solutions




In [ ]:

bit_precisions = [8, 4]
target_model_size = 4.0  # Target model size in MB


sensitivities, parameter_sizes = compute_layer_sensitivity_and_size(model=model, loss_fn =nn.CrossEntropyLoss(ignore_index=255) , 
                                   data_loader=train_loader, epsilon=1e-3, num_samples=10, device='cuda')
# Optimize and visualize
# best_solution, pareto_solutions = dp_optimize(sensitivities, parameter_sizes, bit_precisions, target_model_size)

# Print the best solution
# model_size, accuracy_loss, bit_config = best_solution
# print(f"Best Solution:\nModel Size: {model_size:.2f} MB, Accuracy Loss: {accuracy_loss:.2f}, Bit Precisions: {bit_config}")

# # Print all Pareto-optimal solutions
# print("\nPareto-Optimal Solutions:")
# for size, loss, config in pareto_solutions:
#     print(f"Model Size: {size:.2f} MB, Accuracy Loss: {loss:.2f}, Bit Precisions: {config}")


In [ ]:
parameter_sizes = {'backbone.0.0': 0.0004119873046875,
 'backbone.1.block.0.0': 0.0001373291015625,
 'backbone.1.block.1.0': 0.000244140625,
 'backbone.2.block.0.0': 0.0009765625,
 'backbone.2.block.1.0': 0.00054931640625,
 'backbone.2.block.2.0': 0.00146484375,
 'backbone.3.block.0.0': 0.00164794921875,
 'backbone.3.block.1.0': 0.00061798095703125,
 'backbone.3.block.2.0': 0.00164794921875,
 'backbone.4.block.0.0': 0.00164794921875,
 'backbone.4.block.1.0': 0.00171661376953125,
 'backbone.4.block.2.fc1': 0.00167083740234375,
 'backbone.4.block.2.fc2': 0.00171661376953125,
 'backbone.4.block.3.0': 0.00274658203125,
 'backbone.5.block.0.0': 0.00457763671875,
 'backbone.5.block.1.0': 0.00286102294921875,
 'backbone.5.block.2.fc1': 0.003692626953125,
 'backbone.5.block.2.fc2': 0.00377655029296875,
 'backbone.5.block.3.0': 0.00457763671875,
 'backbone.6.block.0.0': 0.00457763671875,
 'backbone.6.block.1.0': 0.00286102294921875,
 'backbone.6.block.2.fc1': 0.003692626953125,
 'backbone.6.block.2.fc2': 0.00377655029296875,
 'backbone.6.block.3.0': 0.00457763671875,
 'backbone.7.block.0.0': 0.0091552734375,
 'backbone.7.block.1.0': 0.0020599365234375,
 'backbone.7.block.2.0': 0.018310546875,
 'backbone.8.block.0.0': 0.0152587890625,
 'backbone.8.block.1.0': 0.00171661376953125,
 'backbone.8.block.2.0': 0.0152587890625,
 'backbone.9.block.0.0': 0.0140380859375,
 'backbone.9.block.1.0': 0.00157928466796875,
 'backbone.9.block.2.0': 0.0140380859375,
 'backbone.10.block.0.0': 0.0140380859375,
 'backbone.10.block.1.0': 0.00157928466796875,
 'backbone.10.block.2.0': 0.0140380859375,
 'backbone.11.block.0.0': 0.03662109375,
 'backbone.11.block.1.0': 0.004119873046875,
 'backbone.11.block.2.fc1': 0.05504608154296875,
 'backbone.11.block.2.fc2': 0.055389404296875,
 'backbone.11.block.3.0': 0.05126953125,
 'backbone.12.block.0.0': 0.07177734375,
 'backbone.12.block.1.0': 0.005767822265625,
 'backbone.12.block.2.fc1': 0.10782623291015625,
 'backbone.12.block.2.fc2': 0.108306884765625,
 'backbone.12.block.3.0': 0.07177734375,
 'backbone.13.block.0.0': 0.07177734375,
 'backbone.13.block.1.0': 0.016021728515625,
 'backbone.13.block.2.fc1': 0.10782623291015625,
 'backbone.13.block.2.fc2': 0.108306884765625,
 'backbone.13.block.3.0': 0.1025390625,
 'backbone.14.block.0.0': 0.146484375,
 'backbone.14.block.1.0': 0.02288818359375,
 'backbone.14.block.2.fc1': 0.2199554443359375,
 'backbone.14.block.2.fc2': 0.22064208984375,
 'backbone.14.block.3.0': 0.146484375,
 'backbone.15.block.0.0': 0.146484375,
 'backbone.15.block.1.0': 0.02288818359375,
 'backbone.15.block.2.fc1': 0.2199554443359375,
 'backbone.15.block.2.fc2': 0.22064208984375,
 'backbone.15.block.3.0': 0.146484375,
 'backbone.16.0': 0.146484375,
 'classifier.cbr.0': 0.1171875,
 'classifier.scale.1': 0.1171875,
 'classifier.low_classifier': 0.0007429122924804688,
 'classifier.high_classifier': 0.0023374557495117188}

In [ ]:
sensitivities = {'backbone.0.0': 0.5361928880214691,
 'backbone.1.block.0.0': 0.5019792884588241,
 'backbone.1.block.1.0': 0.04563789665699005,
 'backbone.2.block.0.0': 0.6223258435726166,
 'backbone.2.block.1.0': 0.7686699986457824,
 'backbone.2.block.2.0': 0.48881399631500244,
 'backbone.3.block.0.0': 0.4815409779548645,
 'backbone.3.block.1.0': 0.7167946040630341,
 'backbone.3.block.2.0': 1.4325122356414794,
 'backbone.4.block.0.0': 0.4848864495754242,
 'backbone.4.block.1.0': 0.6317842602729797,
 'backbone.4.block.2.fc1': 0.1427037892397493,
 'backbone.4.block.2.fc2': 1.2090101532638073,
 'backbone.4.block.3.0': 1.8327101588249206,
 'backbone.5.block.0.0': 0.6403753757476807,
 'backbone.5.block.1.0': 0.7930723071098328,
 'backbone.5.block.2.fc1': 0.18774263453669846,
 'backbone.5.block.2.fc2': 0.5620469726622105,
 'backbone.5.block.3.0': 1.2488696217536925,
 'backbone.6.block.0.0': 0.7900815010070801,
 'backbone.6.block.1.0': 0.7612608253955842,
 'backbone.6.block.2.fc1': 0.30064537972211836,
 'backbone.6.block.2.fc2': 0.46714989077299834,
 'backbone.6.block.3.0': 1.1708393335342406,
 'backbone.7.block.0.0': 7.91571455001831,
 'backbone.7.block.1.0': 1.217163610458374,
 'backbone.7.block.2.0': 9.78160581588745,
 'backbone.8.block.0.0': 1.3033080577850342,
 'backbone.8.block.1.0': 0.5885966002941132,
 'backbone.8.block.2.0': 5.19773497581482,
 'backbone.9.block.0.0': 1.1723219513893128,
 'backbone.9.block.1.0': 0.5304490208625794,
 'backbone.9.block.2.0': 18.47936668395996,
 'backbone.10.block.0.0': 2.5445281267166138,
 'backbone.10.block.1.0': 0.5079378962516785,
 'backbone.10.block.2.0': 5.4564440727233885,
 'backbone.11.block.0.0': 7.022655057907104,
 'backbone.11.block.1.0': 1.416386640071869,
 'backbone.11.block.2.fc1': 1.3779880344867705,
 'backbone.11.block.2.fc2': 1.726470647379756,
 'backbone.11.block.3.0': 6.657061672210693,
 'backbone.12.block.0.0': 6.7070292949676515,
 'backbone.12.block.1.0': 4.184150886535645,
 'backbone.12.block.2.fc1': 1.8763200944289564,
 'backbone.12.block.2.fc2': 2.8973055623471735,
 'backbone.12.block.3.0': 7.958397102355957,
 'backbone.13.block.0.0': 13.630026721954346,
 'backbone.13.block.1.0': 3.4793856143951416,
 'backbone.13.block.2.fc1': 2.353938566148281,
 'backbone.13.block.2.fc2': 3.795076295733452,
 'backbone.13.block.3.0': 9.370883369445801,
 'backbone.14.block.0.0': 7.938345861434937,
 'backbone.14.block.1.0': 6.600805187225342,
 'backbone.14.block.2.fc1': 15.750698700547218,
 'backbone.14.block.2.fc2': 7.817631259560585,
 'backbone.14.block.3.0': 13.06938123703003,
 'backbone.15.block.0.0': 11.81272087097168,
 'backbone.15.block.1.0': 2.3775880575180053,
 'backbone.15.block.2.fc1': 4.119218256697058,
 'backbone.15.block.2.fc2': 5.775940592586994,
 'backbone.15.block.3.0': 8.030117797851563,
 'backbone.16.0': 6.381353807449341,
 'classifier.cbr.0': 13.620604515075684,
 'classifier.scale.1': 0.038309919834136966,
 'classifier.low_classifier': 0.2214799209497869,
 'classifier.high_classifier': 0.8468556581996382}}

In [ ]:
import matplotlib.pyplot as plt


def visualize_sensitivity_scores(sensitivities, figsize=(20, 8)):
    """
    Visualize sensitivity scores for model layers.

    Args:
        sensitivity_scores (dict): A dictionary where keys are layer names and values are sensitivity scores.
        log_scale (bool): Whether to plot sensitivity scores in log10 scale. Default is True.
        figsize (tuple): Figure size for the plot. Default is (20, 8).

    Returns:
        None
    """
    # Extract layer names and sensitivity scores
    layers = list(sensitivities.keys())
    scores = list(sensitivities.values())
    
     # Create the plot
    plt.figure(figsize=figsize)
    plt.plot(scores, marker='o', linestyle='-', color='b', label='Sensitivity Scores')
    plt.yscale('log')  # Use logarithmic scale for the Y-axis
    plt.xticks(ticks=range(len(layers)), labels=layers, rotation=90, fontsize=8)
    plt.ylabel("Sensitivity Score (Log Scale)")
    plt.xlabel("Layers")
    plt.title("Sensitivity Scores by Layer (Logarithmic Scale)")
    plt.grid(alpha=0.5)
    plt.legend()
    plt.tight_layout()
    plt.show()


# Visualize sensitivity scores
visualize_sensitivity_scores(sensitivities)

In [ ]:
def assign_bit_precision_two_levels(sensitivity_scores, bit_precisions=(8, 4), threshold=0.5):
    """
    Assign 8-bit or 4-bit precision to layers based on sensitivity scores.

    Args:
        sensitivity_scores (dict): Layer sensitivity scores as {layer_name: sensitivity_score}.
        bit_precisions (tuple): Bit precision levels (e.g., 8, 4).
        threshold (float): Normalized threshold for assigning bit precision.

    Returns:
        dict: Mapping of layer names to assigned bit precision.
    """
    # Normalize sensitivity scores
    max_score = max(sensitivity_scores.values())
    min_score = min(sensitivity_scores.values())
    normalized_scores = {
        layer: (score - min_score) / (max_score - min_score)
        for layer, score in sensitivity_scores.items()
    }

    # Assign bit precision based on threshold
    precision_mapping = {}
    for layer, norm_score in normalized_scores.items():
        if norm_score >= threshold:  # High sensitivity
            precision_mapping[layer] = bit_precisions[0]  # 8-bit
        else:  # Low sensitivity
            precision_mapping[layer] = bit_precisions[1]  # 4-bit

    return precision_mapping


# Example Usage
thresholds = [0.2, 0.3, 0.4]  # Conservative, Balanced, Aggressive thresholds
bit_precisions = (8, 4)

for threshold in thresholds:
    print(f"\nExperimenting with Threshold: {threshold}")
    bit_precision_mapping = assign_bit_precision_two_levels(
        sensitivity_scores=sensitivities, 
        bit_precisions=bit_precisions, 
        threshold=threshold
    )
    # Print bit precision mapping for this threshold
    for layer, precision in bit_precision_mapping.items():
        print(f"{layer}: {precision}-bit")


In [ ]:
# Normalize sensitivity scores
max_score = max(sensitivities.values())
min_score = min(sensitivities.values())
normalized_scores = {
    layer: (score - min_score) / (max_score - min_score)
    for layer, score in sensitivities.items()
}

# Visualize normalized scores
import matplotlib.pyplot as plt
plt.hist(normalized_scores.values(), bins=20)
plt.xlabel("Normalized Sensitivity Scores")
plt.ylabel("Frequency")
plt.title("Distribution of Normalized Sensitivity Scores")
plt.show()


In [ ]:
!/usr/src/tensorrt/bin/trtexec

In [ ]:
# Exporting to ONNX
dummy_input = torch.randn(1, 3, 512, 1024, device='cuda')
input_names = ["actual_input_1"]
output_names = ["output1"]
torch.onnx.export(
    model,
    dummy_input,
    "/kaggle/working/deeplab_resnet50_base.onnx",  # Single .onnx extension
    verbose=False,
    opset_version=11,
    do_constant_folding=False,
    input_names=input_names,
    output_names=output_names
)

# Converting ONNX model to TRT
! /usr/src/tensorrt/bin/trtexec --onnx=/kaggle/working/deeplab_resnet50_base.onnx --useSpinWait --avgRuns=100 --fp32  --saveEngine=/kaggle/working/deeplab_resnet50_base.trt 

In [ ]:
sensitivities

In [ ]:
import torch
import numpy as np

# Define raw input tensor (NO normalization applied)
dummy_input = torch.randn(1, 3, 512, 1024, device='cuda')  # Batch size 1 for flexibility

# Define ONNX input and output names
input_names = ["actual_input_1"]
output_names = ["output1"]

# Export the model to ONNX
torch.onnx.export(
    model, 
    dummy_input, 
    "/kaggle/working/mobilenetv3_base.onnx",
    verbose=True,
    opset_version=11,
    do_constant_folding=True,  # Enables constant folding optimization
    input_names=input_names,
    output_names=output_names,
    dynamic_axes={"actual_input_1": {0: "batch_size"}}  # Allow dynamic batch sizes
)

print("✅ ONNX Model Exported Successfully!")


# Convert ONNX model to TensorRT with dynamic shapes
!/usr/src/tensorrt/bin/trtexec --onnx=/kaggle/working/mobilenetv3_base.onnx  --useSpinWait --saveEngine=/kaggle/working/mobilenetv3_base.trt


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class ONNXWrapper(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, x):
        output_dict = self.model(x)  # Get dictionary output
        logits = output_dict['out']  # Extract 'out' key
        return F.softmax(logits, dim=1)  # Apply Softmax

# Load model
# model = MyModel()  # Replace with actual model definition
# model.load_state_dict(torch.load("best_model.pth"))
model.eval()  # Set to eval mode

# Wrap model with ONNXWrapper
wrapped_model = ONNXWrapper(model)

# Export ONNX with Softmax included
torch.onnx.export(
    wrapped_model, 
    torch.randn(1, 3, 512, 1024, device='cuda'), 
    "/kaggle/working/mobilenetv3_base.onnx",
    opset_version=11,
    do_constant_folding=True,
    input_names=["actual_input_1"],
    output_names=["output1"],
    dynamic_axes={"actual_input_1": {0: "batch_size"}}
)

print("✅ ONNX Model Exported with Softmax Included!")


In [ ]:
import torch
import numpy as np

# Define normalization (same as dataset pipeline)
mean = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1).cuda()
std = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1).cuda()

# Generate input tensor and apply normalization
dummy_input = torch.randn(16, 3, 512, 1024, device='cuda')
normalized_input = (dummy_input - mean) / std  # Apply normalization

# Define ONNX input and output names
input_names = ["actual_input_1"]
output_names = ["output1"]

# Export the model to ONNX with normalized input
torch.onnx.export(
    model.eval(), 
    normalized_input,  # Pass the normalized input for export
    "/kaggle/working/mobilenetv3_base.onnx",
    verbose=True,
    opset_version=11,
    do_constant_folding=True,
    input_names=input_names,
    output_names=output_names,
    dynamic_axes={"actual_input_1": {0: "batch_size"}}  # Allow dynamic batch sizes
)

print("✅ ONNX Model Exported with Correct Normalization!")


In [ ]:
!python -m onnxsim /kaggle/working/mobilenetv3_base.onnx /kaggle/working/mobilenetv3_base_simplified.onnx


In [ ]:
!export LD_LIBRARY_PATH=/usr/local/cuda-11/lib64:$LD_LIBRARY_PATH
!export CUDNN_VERSION=8.4.1


In [ ]:
import onnxruntime as ort
import numpy as np

# Load ONNX model
onnx_model_path = "/kaggle/working/mobilenetv3_base_simplified.onnx"
ort_session = ort.InferenceSession(onnx_model_path)

# Generate test input
dummy_input = np.random.randn(1, 3, 512, 1024).astype(np.float32)

# Run inference
onnx_output = ort_session.run(None, {"actual_input_1": dummy_input})

# Print sample output values
print("🔹 ONNX Output Stats:")
print("Min:", np.min(onnx_output[0]))
print("Max:", np.max(onnx_output[0]))
print("Mean:", np.mean(onnx_output[0]))
print("Std Dev:", np.std(onnx_output[0]))


In [ ]:
!/usr/src/tensorrt/bin/trtexec --onnx=/kaggle/working/mobilenetv3_base_simplified.onnx \
--saveEngine=/kaggle/working/mobilenetv3_base_fp32.trt \
--useSpinWait \
--workspace=1024 \
--precisionConstraints=obey \
--minShapes=actual_input_1:1x3x512x1024 \
--optShapes=actual_input_1:8x3x512x1024 \
--maxShapes=actual_input_1:16x3x512x1024


In [ ]:
!/usr/src/tensorrt/bin/trtexec --onnx=mobilenetv3_base_simplified.onnx \
--saveEngine=mobilenetv3_base_fp32_fixed.trt \
--useSpinWait \
--precisionConstraints=obey \
--layerPrecisions=Conv:fp32,MatMul:fp32,Add:fp32 \
--layerOutputTypes=Conv:fp32,MatMul:fp32,Add:fp32 \
--noTF32 \
--workspace=1024 \
--inputIOFormats=fp32:chw \
--outputIOFormats=fp32:chw \
--minShapes=actual_input_1:1x3x512x1024 \
--optShapes=actual_input_1:8x3x512x1024 \
--maxShapes=actual_input_1:16x3x512x1024


In [ ]:
!/usr/src/tensorrt/bin/trtexec --onnx=/kaggle/working/mobilenetv3_base_simplified.onnx \
--saveEngine=/kaggle/working/mobilenetv3_base_fp16.trt \
--useSpinWait \
--workspace=1024 \
--fp16 \
--precisionConstraints=obey \
--minShapes=actual_input_1:1x3x512x1024 \
--optShapes=actual_input_1:8x3x512x1024 \
--maxShapes=actual_input_1:16x3x512x1024


In [ ]:
import onnxruntime as ort
import torch

# Load ONNX model
onnx_model_path = "/kaggle/working/mobilenetv3_base_simplified.onnx"
ort_session = ort.InferenceSession(onnx_model_path)

# Create a test input
dummy_input = torch.randn(1, 3, 512, 1024).numpy()

# Run inference
onnx_output = ort_session.run(None, {"actual_input_1": dummy_input})

# Check output shape and values
print(f"ONNX Output Shape: {onnx_output[0].shape}")
print(f"ONNX Output Values: {onnx_output[0][0, :5, :5, :5]}")  # Check sample values


In [ ]:
import tensorrt as trt
import numpy as np
import os

class Int8Calibrator(trt.IInt8EntropyCalibrator2):
    def __init__(self, data_loader, cache_file="calibration_cache.bin"):
        super().__init__()
        self.data_loader = data_loader
        self.cache_file = cache_file
        self.batch_size = 8
        self.device_input = None
        self.current_index = 0
        self.data = None

    def get_batch_size(self):
        return self.batch_size

    def get_batch(self, names):
        if self.current_index >= len(self.data_loader):
            return None

        batch = self.data_loader[self.current_index]
        batch = np.ascontiguousarray(batch.numpy().astype(np.float32))
        self.current_index += 1

        return [batch.ravel()]

    def read_calibration_cache(self):
        if os.path.exists(self.cache_file):
            with open(self.cache_file, "rb") as f:
                return f.read()
        return None

    def write_calibration_cache(self, cache):
        with open(self.cache_file, "wb") as f:
            f.write(cache)


In [ ]:
calibrator = Int8Calibrator(
    data_loader=train_loader,  # Use full dataset for calibration
    cache_file="\kaggle\working\calibration_cache.bin"
)


In [ ]:
!/usr/src/tensorrt/bin/trtexec --onnx=/kaggle/working/mobilenetv3_base_simplified.onnx \
--saveEngine=/kaggle/working/mobilenetv3_base_int8.trt \
--useSpinWait \
--workspace=1024 \
--int8 \
--calib=/kaggle/working/calibration_cache.bin \
--minShapes=actual_input_1:1x3x512x1024 \
--optShapes=actual_input_1:8x3x512x1024 \
--maxShapes=actual_input_1:16x3x512x1024


In [ ]:
!/usr/src/tensorrt/bin/trtexec --loadEngine=/kaggle/working/mobilenetv3_base_fp32_calibrated.trt --dumpLayerInfo


In [ ]:
!/usr/src/tensorrt/bin/trtexec --loadEngine=mobilenetv3_base_fp32.trt --dumpOutput

In [ ]:
import onnxruntime as ort
import numpy as np

# Load ONNX model
onnx_model_path = "/kaggle/working/mobilenetv3_base_simplified.onnx"
ort_session = ort.InferenceSession(onnx_model_path)

# Load real validation image
images, masks = next(iter(val_loader))  # Get first batch
images = images.numpy()

# Run inference on actual validation data
onnx_output = ort_session.run(None, {"actual_input_1": images})[0]

# Print statistics
print("🔹 ONNX Validation Output Stats:")
print("Min:", np.min(onnx_output))
print("Max:", np.max(onnx_output))
print("Mean:", np.mean(onnx_output))
print("Std Dev:", np.std(onnx_output))


In [ ]:
!/usr/src/tensorrt/bin/trtexec --onnx=mobilenetv3_base_simplified.onnx \
--saveEngine=mobilenetv3_base_fp32_calibrated.trt \
--int8 \
--calib=calibration_cache.bin \
--workspace=1024


In [ ]:
import numpy as np
import torch
import time
import tensorrt as trt
import pycuda.driver as cuda
import pycuda.autoinit

def evaluate_tensorrt(engine_path, dataloader, criterion, device, num_classes, batch_size=16):
    """
    Run inference and evaluate the TensorRT model using validation data.
    Computes accuracy, mean IoU, latency, and throughput using the same metrics as PyTorch inference.
    """
    TRT_LOGGER = trt.Logger(trt.Logger.WARNING)
    
    with open(engine_path, "rb") as f, trt.Runtime(TRT_LOGGER) as runtime:
        engine = runtime.deserialize_cuda_engine(f.read())

    context = engine.create_execution_context()
    input_shape = (batch_size, 3, 512, 1024)
    context.set_binding_shape(0, input_shape)

    d_input = cuda.mem_alloc(batch_size * 3 * 512 * 1024 * np.float32().itemsize)
    d_output = cuda.mem_alloc(batch_size * num_classes * 512 * 1024 * np.float32().itemsize)
    h_output = np.empty((batch_size, num_classes, 512, 1024), dtype=np.float32)

    total_loss, total_accuracy, total_iou, latencies = 0.0, 0.0, 0.0, []
    total_time, total_batches = 0.0, 0

    mean = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1).to(device)
    std = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1).to(device)

    with torch.no_grad():
        for images, masks in dataloader:
            if images.size(0) < batch_size:
                break

            images, masks = images[:batch_size].to(device), masks[:batch_size].to(device).long()
            images = (images - mean) / std  # Normalize input

            # Transfer input to GPU
            cuda.memcpy_htod(d_input, images.cpu().numpy().astype(np.float32))

            # Measure latency
            start_time = time.time()
            context.execute_v2(bindings=[int(d_input), int(d_output)])
            cuda.memcpy_dtoh(h_output, d_output)
            end_time = time.time()

            latency = (end_time - start_time) * 1000  # Convert to milliseconds
            latencies.append(latency)
            total_time += (end_time - start_time)
            total_batches += 1

            # Convert output to tensor
            output = torch.from_numpy(h_output).cuda().float()
            preds = output.argmax(dim=1)  # REMOVE Softmax, since ONNX already has it

            # Compute loss
            loss = criterion(output, masks)
            total_loss += loss.item()

            # Compute accuracy
            accuracy = calculate_accuracy(output, masks)
            total_accuracy += accuracy.item()

            # Compute mean IoU
            iou = calculate_mean_iou(preds, masks, num_classes)
            total_iou += iou

            torch.cuda.empty_cache()

    # Compute final metrics
    avg_loss = total_loss / total_batches
    avg_accuracy = total_accuracy / total_batches
    avg_iou = total_iou / total_batches
    avg_latency = np.mean(latencies)
    throughput = (total_batches * batch_size) / total_time  # FPS

    print(f"\n✅ Model: {engine_path}")
    print(f"   Accuracy: {avg_accuracy * 100:.2f}%")
    print(f"   Mean IoU: {avg_iou:.4f}")
    print(f"   Avg Latency: {avg_latency:.2f} ms")
    print(f"   Throughput: {throughput:.2f} samples/sec")

    return avg_accuracy, avg_iou, avg_latency, throughput


In [ ]:
# Load validation dataset (assuming it's already defined)
# val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False, num_workers=4, pin_memory=True)

# Evaluate FP32
fp32_acc, fp32_iou, fp32_latency, fp32_fps = evaluate_tensorrt(
    "/kaggle/working/mobilenetv3_base_fp32.trt", val_loader, criterion, device, num_classes, batch_size=16
)

#Evaluate FP16
fp16_acc, fp16_iou, fp16_latency, fp16_fps = evaluate_tensorrt(
    "/kaggle/working/mobilenetv3_base_fp16.trt", val_loader, criterion, device, num_classes, batch_size=16
)

# Evaluate INT8
int8_acc, int8_iou, int8_latency, int8_fps = evaluate_tensorrt(
    "/kaggle/working/mobilenetv3_base_int8.trt", val_loader, criterion, device, num_classes, batch_size=16
)

# Print final results
print("\n🔹 **Final Comparison** 🔹")
print(f"{'Precision':<10} | {'Accuracy (%)':<12} | {'mIoU':<8} | {'Latency (ms)':<14} | {'FPS':<10}")
print("-" * 60)
print(f"{'FP32':<10} | {fp32_acc * 100:.2f}       | {fp32_iou:.4f} | {fp32_latency:.2f} ms     | {fp32_fps:.2f} FPS")
print(f"{'FP16':<10} | {fp16_acc * 100:.2f}       | {fp16_iou:.4f} | {fp16_latency:.2f} ms     | {fp16_fps:.2f} FPS")
print(f"{'INT8':<10} | {int8_acc * 100:.2f}       | {int8_iou:.4f} | {int8_latency:.2f} ms     | {int8_fps:.2f} FPS")


In [ ]:
fp32_acc, fp32_iou, fp32_latency, fp32_fps = evaluate_tensorrt(
    "/kaggle/working/base_fp32.engine", val_loader, criterion, device, num_classes, batch_size=16
)


In [ ]:
fp32_acc, fp32_iou, fp32_latency, fp32_fps = evaluate_tensorrt(
    "/kaggle/working/base_fp32.engine", inference_loader, criterion, device, num_classes, batch_size=1
)


In [ ]:
import onnxruntime as ort
import torch

# Load ONNX model
onnx_model_path = "/kaggle/working/mobilenetv3_base_simplified.onnx"
ort_session = ort.InferenceSession(onnx_model_path)

# Create a test input
dummy_input = torch.randn(1, 3, 512, 1024).numpy()

# Run inference
onnx_output = ort_session.run(None, {"actual_input_1": dummy_input})

# Check output shape
print(f"ONNX Output Shape: {onnx_output[0].shape}")


In [ ]:
import torch

# Define normalization parameters
mean = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1).cuda()  # For PyTorch
std = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1).cuda()

# Generate dynamic input tensor and normalize
dummy_input = torch.randn(16, 3, 512, 1024, device='cuda')  # Dynamic size
normalized_input = (dummy_input - mean) / std
# Specify input and output names
input_names = ["actual_input_1"]
output_names = ["output1"]

# Export the model with dynamic input sizes
torch.onnx.export(
    model,
    dummy_input,
    "/kaggle/working/mobilenetv3_half.onnx",  # Save as .onnx file
    verbose=False,
    opset_version=11,
    do_constant_folding=False,
    input_names=input_names,
    output_names=output_names,
    # dynamic_axes=dynamic_axes  # Specify dynamic axes
)

# Convert ONNX model to TensorRT with dynamic shapes
!/usr/src/tensorrt/bin/trtexec --onnx=/kaggle/working/mobilenetv3_half.onnx  --fp16 --precisionConstraints=prefer --useSpinWait --saveEngine=/kaggle/working/mobilenetv3_half.trt


In [ ]:
### import torch

# Define dummy input with a fixed size for export, but specify dynamic dimensions
# Define normalization parameters
mean = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1).cuda()  # For PyTorch
std = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1).cuda()

# Generate dynamic input tensor and normalize
dummy_input = torch.randn(16, 3, 512, 1024, device='cuda')  # Dynamic size
normalized_input = (dummy_input - mean) / std

# Specify input and output names
input_names = ["actual_input_1"]
output_names = ["output1"]

# Export the model with dynamic input sizes
torch.onnx.export(
    model,
    dummy_input,
    "/kaggle/working/mobilenetv3_int8.onnx",  # Save as .onnx file
    verbose=False,
    opset_version=11,
    do_constant_folding=False,
    input_names=input_names,
    output_names=output_names,
    # dynamic_axes=dynamic_axes  # Specify dynamic axes
)

# Convert ONNX model to TensorRT with dynamic shapes
!/usr/src/tensorrt/bin/trtexec --onnx=/kaggle/working/mobilenetv3_int8.onnx --int8 --calib=calibration_data.cache --useSpinWait --workspace=512 --saveEngine=/kaggle/working/mobilenetv3_int8.trt


In [ ]:
#-----------per channel weight-----------------------------------
# quant_desc_input = QuantDescriptor(num_bits=8, calib_method='max', axis=0)
# quant_nn.QuantConv2d.set_default_quant_desc_input(quant_desc_input)
# quant_nn.QuantMaxPool2d.set_default_quant_desc_input(quant_desc_input)
# quant_nn.QuantAdaptiveAvgPool2d.set_default_quant_desc_input(quant_desc_input)
# quant_nn.QuantLinear.set_default_quant_desc_input(quant_desc_input)

# #-----------per tensor weight--------------------------------------
quant_desc_input = QuantDescriptor(num_bits=8, calib_method='entropy', axis=None)
quant_nn.QuantConv2d.set_default_quant_desc_input(quant_desc_input)
# quant_nn.QuantMaxPool2d.set_default_quant_desc_input(quant_desc_input)
# quant_nn.QuantAdaptiveAvgPool2d.set_default_quant_desc_input(quant_desc_input)
# quant_nn.QuantConvTranspose2d.set_default_quant_desc_input(quant_desc_input)
quant_nn.QuantLinear.set_default_quant_desc_input(quant_desc_input)


quant_desc_weight = QuantDescriptor(num_bits=8, calib_method='entropy', axis=0)
quant_nn.QuantConv2d.set_default_quant_desc_weight(quant_desc_weight)
# quant_nn.QuantConvTranspose2d.set_default_quant_desc_weight(quant_desc_weight)
# quant_nn.QuantAdaptiveAvgPool2d.set_default_quant_desc_input(quant_desc_weight)
quant_nn.QuantLinear.set_default_quant_desc_weight(quant_desc_weight)



In [ ]:
from pytorch_quantization.tensor_quant import QuantDescriptor
from pytorch_quantization.nn import QuantConv2d, QuantAdaptiveAvgPool2d
import torch.nn as nn

def apply_selective_fake_quantization(model):
    """
    Apply selective fake quantization to the given model by targeting robust layers
    and skipping sensitive ones.

    Args:
        model (torch.nn.Module): The PyTorch model to modify with fake quantization.

    Returns:
        torch.nn.Module: The model with selective fake quantization applied.
    """
    # Define QuantDescriptors
    conv_quant_desc = QuantDescriptor(num_bits=8, axis=0, calib_method="max")  # Per-channel for weights
    activation_quant_desc = QuantDescriptor(num_bits=8, axis=None, calib_method="max")  # Per-tensor for activations

    # Function to apply fake quantization to specific layers
    def apply_quantization_to_layer(layer):
        if isinstance(layer, QuantConv2d):
            # Apply fake quantizers to weights and inputs
            layer._weight_quantizer.set_quant_desc(conv_quant_desc)
            layer._input_quantizer.set_quant_desc(activation_quant_desc)
        elif isinstance(layer, QuantAdaptiveAvgPool2d):
            # Optional: Apply or skip fake quantization for pooling layers
            layer._input_quantizer.set_quant_desc(activation_quant_desc)

    # Function to skip quantization for sensitive layers
    def skip_quantization_for_layer(layer):
        if hasattr(layer, "_input_quantizer"):
            layer._input_quantizer = None
        if hasattr(layer, "_weight_quantizer"):
            layer._weight_quantizer = None

    # Iterate over all modules and selectively apply fake quantization
    for name, module in model.named_modules():
        if isinstance(module, (QuantConv2d, QuantAdaptiveAvgPool2d)):
            apply_quantization_to_layer(module)
        elif isinstance(module, (nn.BatchNorm2d, nn.SiLU, nn.Hardswish)):
            skip_quantization_for_layer(module)
        elif isinstance(module, nn.ModuleList):
            # Recursively handle nested modules
            for sub_module in module:
                apply_selective_fake_quantization(sub_module)

    return model


In [ ]:
quant_modules.initialize()

quant_modules.initialize()

In [ ]:
q_model = lraspp_mobilenet_v3_large(pretrained=False) # Load the model pre-trained on COCO
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

num_classes = 19  # Cityscapes has 19 valid classes
# Modify low_classifier and high_classifier for 19 classes
# Assuming `model` is your LRASPP-based model
replace_relu_with_silu(q_model)
q_model.classifier.low_classifier = torch.nn.Conv2d(40, 19, kernel_size=1, stride=1)
q_model.classifier.high_classifier = torch.nn.Conv2d(128, 19, kernel_size=1, stride=1)
q_model = q_model.to(device)

In [ ]:
if torch.cuda.device_count() > 1:
    print("Using", torch.cuda.device_count(), "GPUs!")
    q_model = nn.DataParallel(q_model)  # Wrap model in DataParallel


In [ ]:
def disable_quantization_for_adaptive_pooling(model):
    for name, module in model.named_modules():
        # Check for AdaptiveAvgPool2d or QuantAdaptiveAvgPool2d layers
        if isinstance(module, quant_nn.QuantAdaptiveAvgPool2d):
            print(f"Disabling quantization for layer: {name}")
            module._input_quantizer.disable()  # Disable input quantization
            if hasattr(module, "_input_quantizer"):
                module._input_quantizer.disable()  # Disable weight quantization if present

disable_quantization_for_adaptive_pooling(q_model)

In [ ]:
q_model

In [ ]:
# Load the checkpoint from the specified path
checkpoint_path = "/kaggle/input/lsarpp_mobilenetv3_segmentation/pytorch/v2/2/lraspp_mobilenetv3_v2.pth"
checkpoint = torch.load(checkpoint_path)

# Explicitly process the state_dict keys
state_dict = checkpoint['model_state_dict']
modified_state_dict = {}
for key, val in state_dict.items():
    # Check if the key starts with 'module.' and remove it
    if key.startswith('module.'):
        modified_state_dict[key[7:]] = val  # Remove the 'module.' prefix
    else:
        modified_state_dict[key] = val  # Keep the key as it is

# Load the processed state_dict into the model
q_model.load_state_dict(modified_state_dict)

# Load the optimizer state if needed
if 'optimizer_state_dict' in checkpoint:
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])

# Retrieve additional metadata (epoch and validation accuracy)
epoch = checkpoint.get('epoch', 'unknown')
val_accuracy = checkpoint.get('val_accuracy', None)
if val_accuracy is not None:
    print(f"Model loaded from epoch {epoch} with validation accuracy {val_accuracy * 100:.2f}%")
else:
    print(f"Model loaded from epoch {epoch}.")

# Ensure the model is on the correct device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
q_model = q_model.to(device)

# Set the model to evaluation mode for inference
q_model.eval()


In [ ]:
q_model

In [ ]:
def compute_amax(model, **kwargs):
    # Load calib result
    for name, module in model.named_modules():
        if isinstance(module, quant_nn.TensorQuantizer):
            if module._calibrator is not None:
                if isinstance(module._calibrator, calib.MaxCalibrator):
                    module.load_calib_amax()
                else:
                    module.load_calib_amax(**kwargs)
            print(F"{name:40}: {module}")
    model.cuda()

def collect_stats(model, data_loader, num_batches):
    """Feed data to the network and collect statistics"""
    # Enable calibrators
    for name, module in model.named_modules():
        if isinstance(module, quant_nn.TensorQuantizer):
            if module._calibrator is not None:
                module.disable_quant()
                module.enable_calib()
            else:
                module.disable()

    # Feed data to the network for collecting stats
    for i, (image, _) in tqdm(enumerate(data_loader), total=num_batches):
        model(image.cuda())
        if i >= num_batches:
            break

    # Disable calibrators
    for name, module in model.named_modules():
        if isinstance(module, quant_nn.TensorQuantizer):
            if module._calibrator is not None:
                module.enable_quant()
                module.disable_calib()
            else:
                module.enable()

def calibrate_model(model, model_name, data_loader, num_calib_batch, calibrator, hist_percentile, out_dir):
    """
        Feed data to the network and calibrate.
        Arguments:
            model: classification model
            model_name: name to use when creating state files
            data_loader: calibration data set
            num_calib_batch: amount of calibration passes to perform
            calibrator: type of calibration to use (max/histogram)
            hist_percentile: percentiles to be used for historgram calibration
            out_dir: dir to save state files in
    """

    if num_calib_batch > 0:
        print("Calibrating model")
        with torch.no_grad():
            collect_stats(model, data_loader, num_calib_batch)a

        if not calibrator == "histogram":
            compute_amax(model, method="max")
            calib_output = os.path.join(
                out_dir,
                F"{model_name}-max-{num_calib_batch*data_loader.batch_size}.pth")
            torch.save(model.state_dict(), calib_output)
        else:
            for percentile in hist_percentile:
                print(F"{percentile} percentile calibration")
                compute_amax(model, method="percentile")
                calib_output = os.path.join(
                    out_dir,
                    F"{model_name}-percentile-{percentile}-{num_calib_batch*data_loader.batch_size}.pth")
                torch.save(model.state_dict(), calib_output)

            for method in ["entropy"]:
                print(F"{method} calibration")
                compute_amax(model, method=method)
                calib_output = os.path.join(
                    out_dir,
                    F"{model_name}-{method}-{num_calib_batch*data_loader.batch_size}.pth")
                torch.save(model.state_dict(), calib_output)

In [ ]:
!mkdir calib_cache

In [ ]:
q_model

In [ ]:
#Calibrate the model using max calibration technique.
with torch.no_grad():
    calibrate_model(
        model=q_model,
        model_name="lraspp_mobilenetv3_calib",
        data_loader=train_loader,
        num_calib_batch=32,
        calibrator="max",
        hist_percentile=[99.9], 
        out_dir="/kaggle/working/calib_cache/")
"""[ 99.99, 99.999, 99.9999]"""

In [ ]:
# Set static member of TensorQuantizer to use Pytorch’s own fake quantization functions
quant_nn.TensorQuantizer.use_fb_fake_quant = True

# Exporting to ONNX
dummy_input = torch.randn(1, 3, 512, 1024, device='cuda')
input_names = ["actual_input_1"]
output_names = ["output1"]
torch.onnx.export(
    model,
    dummy_input,
    "/kaggle/working/mobilenetv3_ptq.onnx",  # Single .onnx extension
    verbose=False,
    opset_version=11,
    do_constant_folding=False,
    input_names=input_names,
    output_names=output_names
)

# Converting ONNX model to TRT
! /usr/src/tensorrt/bin/trtexec --onnx=/kaggle/working/mobilenetv3_ptq.onnx --useSpinWait --avgRuns=100  --saveEngine=/kaggle/working/mobilenetv3_ptq.trt

In [ ]:
val_loss, val_accuracy, val_iou = evaluate_model_with_metrics(q_model, val_loader, criterion, device, num_classes)
print(f"Validation Loss: {val_loss:.4f}, Validation Acc: {val_accuracy:.4f}, Validation IoU: {val_iou:.4f}")

In [ ]:
import os


optimizer = optim.AdamW(q_model.parameters(), lr=2e-4, weight_decay=1e-3)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=2, threshold=0.001, verbose=True)
# Cosine Annealing Scheduler
T_max = 10  # Total number of epochs for the cosine curve
# scheduler = CosineAnnealingLR(optimizer, T_max=T_max, eta_min=1e-4)  # eta_min is the minimum LR

# Gradient Clipping (during training loop)
# torch.nn.utils.clip_grad_norm_(q_model.parameters(), max_norm=1.0)

num_epochs = 10
patience = 5  # Stop training if no improvement for 5 epochs
early_stopping = EarlyStopping(patience=patience, min_delta=0.01)
best_val_accuracy = 0
log_path = '/kaggle/working/training_log_mp.txt'

# if accelerator.is_main_process:
    # Open log file only on the main process
with open(log_path, 'w') as log_file:
    log_file.write("Epoch, Train Loss, Train Acc, Val Loss, Val Acc, Val IoU\n")

# Training loop with early stopping
for epoch in range(num_epochs):
    # Train the model for one epoch
    train_loss, train_accuracy = train_one_epoch(q_model, train_loader, criterion, optimizer, device)
    val_loss, val_accuracy, val_iou = evaluate_model_with_metrics(q_model, val_loader, criterion, device, num_classes)

    # Print metrics for the current epoch (on main process only)
    # if accelerator.is_main_process:
    print(f"Epoch {epoch + 1}/{num_epochs}")
    print(f"Train Loss: {train_loss:.4f}, Train Acc: {train_accuracy:.4f}")
    print(f"Validation Loss: {val_loss:.4f}, Validation Acc: {val_accuracy:.4f}, Validation IoU: {val_iou:.4f}")
    # Step the scheduler if using a plateau scheduler
    scheduler.step()
    # Log metrics to file
    with open(log_path, 'a') as log_file:
        log_file.write(f"{epoch + 1}, {train_loss:.4f}, {train_accuracy:.4f}, {val_loss:.4f}, {val_accuracy:.4f}, {val_iou:.4f}\n")

    # Save the model checkpoint if validation accuracy improves
    if val_accuracy > best_val_accuracy:
        best_val_accuracy = val_accuracy
        save_checkpoint(q_model, optimizer, epoch, val_accuracy, path='/kaggle/working/lasrpp_mobilenetv3_qat_mp_a4.pth')

    # Early stopping
    early_stopping(val_accuracy)
    if early_stopping.early_stop:
        # if accelerator.is_main_process:
        print("Early stopping triggered. Stopping training.")
        break

    

# if accelerator.is_main_process:
print("Training complete!")


In [ ]:
# Set static member of TensorQuantizer to use Pytorch’s own fake quantization functions
quant_nn.TensorQuantizer.use_fb_fake_quant = True

# Define normalization parameters
mean = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1).cuda()  # For PyTorch
std = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1).cuda()

# Generate dynamic input tensor and normalize
dummy_input = torch.randn(1, 3, 512, 1024, device='cuda')  # Dynamic size
normalized_input = (dummy_input - mean) / std

# Specify input and output names
input_names = ["actual_input_1"]
output_names = ["output1"]

torch.onnx.export(
    q_model.eval(),
    dummy_input,
    "/kaggle/working/mobilenetv3_qat.onnx",  # Single .onnx extension
    verbose=False,
    opset_version=13,
    do_constant_folding=False,
    input_names=input_names,
    output_names=output_names
)

# Converting ONNX model to TRT
! /usr/src/tensorrt/bin/trtexec --onnx=/kaggle/working/mobilenetv3_qat.onnx --int8 --useSpinWait --avgRuns=100  --saveEngine=/kaggle/working/mobilenetv3_qat.trt

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from pytorch_quantization import nn as quant_nn
from pytorch_quantization.tensor_quant import QuantDescriptor

class QuantConv2d(nn.Conv2d):
    def __init__(self, in_channels, out_channels, kernel_size, stride=1, padding=0, dilation=1, groups=1, bias=True):
        super(QuantConv2d, self).__init__(in_channels, out_channels, kernel_size, stride, padding, dilation, groups, bias)
        # Initialize input and weight quantizers with default 8-bit precision
        self.input_quantizer = TensorQuantizer(QuantDescriptor(num_bits=8, calib_method='max', learn_amax=True))
        self.weight_quantizer = TensorQuantizer(QuantDescriptor(num_bits=8, calib_method='max', learn_amax=True))

    def forward(self, input):
        # Quantize the input and weights before applying convolution
        quant_input = self.input_quantizer(input)
        quant_weight = self.weight_quantizer(self.weight)
        return F.conv2d(quant_input, quant_weight, self.bias, self.stride, self.padding, self.dilation, self.groups)

    def update_num_bits(self, num_bits_input, num_bits_weight):
        # Update the bit-width for input and weight quantizers
        self.input_quantizer.num_bits = num_bits_input
        self.weight_quantizer.num_bits = num_bits_weight


class QuantLinear(nn.Linear):
    def __init__(self, in_features, out_features, bias=True):
        super(QuantLinear, self).__init__(in_features, out_features, bias)
        # Initialize input and weight quantizers with default 8-bit precision
        self.input_quantizer = TensorQuantizer(QuantDescriptor(num_bits=8, calib_method='max', learn_amax=True))
        self.weight_quantizer = TensorQuantizer(QuantDescriptor(num_bits=8, calib_method='max', learn_amax=True))

    def forward(self, input):
        # Quantize the input and weights before applying the linear transformation
        quant_input = self.input_quantizer(input)
        quant_weight = self.weight_quantizer(self.weight)
        return F.linear(quant_input, quant_weight, self.bias)

    def update_num_bits(self, num_bits_input, num_bits_weight):
        # Update the bit-width for input and weight quantizers
        self.input_quantizer.num_bits = num_bits_input
        self.weight_quantizer.num_bits = num_bits_weight


In [ ]:
def update_quantization_bits_with_configs(model, configs, parent_name=""):
    """
    Update the quantization bit-widths of specific layers based on a configuration dictionary.
    
    Args:
        model (nn.Module): The model to update.
        configs (dict): Dictionary where keys are layer names and values are desired bit-widths.
                        Example: {'conv1': 8, 'layer1.0.conv1': 8, 'layer1.0.conv2': 4}
        parent_name (str): The name of the parent module (for recursive tracking).
    
    Returns:
        nn.Module: The updated model with the appropriate bit-widths applied.
    """
    for name, module in model.named_children():
        # Construct the full layer name for nested layers
        layer_name = f"{parent_name}.{name}" if parent_name else name

        # Check if the layer_name exists in the configs dict
        if layer_name in configs:
            num_bits = configs[layer_name]

            # Update bits if the module has quantizers
            if hasattr(module, 'input_quantizer') and isinstance(module.input_quantizer, TensorQuantizer):
                module.input_quantizer.num_bits = 4
                print(f"Updated input quantizer for {layer_name} to {num_bits}-bit.")
            
            if hasattr(module, 'weight_quantizer') and isinstance(module.weight_quantizer, TensorQuantizer):
                module.weight_quantizer.num_bits = num_bits
                print(f"Updated weight quantizer for {layer_name} to {num_bits}-bit.")

        # Recurse into child modules if they exist
        if list(module.children()):  # Check if module has children
            update_quantization_bits_with_configs(module, configs, layer_name)
    
    return model


In [ ]:
q_model = update_quantization_bits_with_configs(q_model, configs, parent_name="")

In [ ]:
#conservative - 0.2 threshold
configs = {
    "backbone.0.0": 4,
"backbone.1.block.0.0": 4,
"backbone.1.block.1.0": 4,
"backbone.2.block.0.0": 4,
"backbone.2.block.1.0": 4,
"backbone.2.block.2.0": 4,
"backbone.3.block.0.0": 4,
"backbone.3.block.1.0": 4,
"backbone.3.block.2.0": 4,
"backbone.4.block.0.0": 4,
"backbone.4.block.1.0": 4,
"backbone.4.block.2.fc1": 4,
"backbone.4.block.2.fc2": 4,
"backbone.4.block.3.0": 4,
"backbone.5.block.0.0": 4,
"backbone.5.block.1.0": 4,
"backbone.5.block.2.fc1": 4,
"backbone.5.block.2.fc2": 4,
"backbone.5.block.3.0": 4,
"backbone.6.block.0.0": 4,
"backbone.6.block.1.0": 4,
"backbone.6.block.2.fc1": 4,
"backbone.6.block.2.fc2": 4,
"backbone.6.block.3.0": 4,
"backbone.7.block.0.0": 8,
"backbone.7.block.1.0": 4,
"backbone.7.block.2.0": 8,
"backbone.8.block.0.0": 4,
"backbone.8.block.1.0": 4,
"backbone.8.block.2.0": 8,
"backbone.9.block.0.0": 4,
"backbone.9.block.1.0": 4,
"backbone.9.block.2.0": 8,
"backbone.10.block.0.0": 4,
"backbone.10.block.1.0": 4,
"backbone.10.block.2.0": 8,
"backbone.11.block.0.0": 8,
"backbone.11.block.1.0": 4,
"backbone.11.block.2.fc1": 4,
"backbone.11.block.2.fc2": 4,
"backbone.11.block.3.0": 8,
"backbone.12.block.0.0": 8,
"backbone.12.block.1.0": 8,
"backbone.12.block.2.fc1": 4,
"backbone.12.block.2.fc2": 4,
"backbone.12.block.3.0": 8,
"backbone.13.block.0.0": 8,
"backbone.13.block.1.0": 4,
"backbone.13.block.2.fc1": 4,
"backbone.13.block.2.fc2": 8,
"backbone.13.block.3.0": 8,
"backbone.14.block.0.0": 8,
"backbone.14.block.1.0": 8,
"backbone.14.block.2.fc1": 8,
"backbone.14.block.2.fc2": 8,
"backbone.14.block.3.0": 8,
"backbone.15.block.0.0": 8,
"backbone.15.block.1.0": 4,
"backbone.15.block.2.fc1": 8,
"backbone.15.block.2.fc2": 8,
"backbone.15.block.3.0": 8,
"backbone.16.0": 8,
"classifier.cbr.0": 8,
"classifier.scale.1": 4,
"classifier.low_classifier": 4,
"classifier.high_classifier": 4,
}

In [ ]:
import time
import tensorrt as trt
import pycuda.driver as cuda
import pycuda.autoinit

def calculate_throughput(model, input_shape=(8, 3, 512, 1024), nwarmup=50, nruns=100):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device).eval()
    input_data = torch.randn(input_shape).to(device)
    
    # Warm-up
    with torch.no_grad():
        for _ in range(nwarmup):
            _ = model(input_data)
    torch.cuda.synchronize()
    
    # Measure throughput
    timings = []
    with torch.no_grad():
        for _ in range(nruns):
            start_time = time.time()
            _ = model(input_data)
            torch.cuda.synchronize()
            end_time = time.time()
            timings.append(end_time - start_time)
    
    avg_batch_time = sum(timings) / nruns
    batch_size = input_shape[0]
    throughput = batch_size / avg_batch_time
    print(f"Average Throughput: {throughput:.2f} samples/second")
    return throughput


def calculate_latency(model, input_shape=(1, 3, 512, 1024), nwarmup=50, nruns=100):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device).eval()
    input_data = torch.randn(input_shape).to(device)
    
    # Warm-up
    with torch.no_grad():
        for _ in range(nwarmup):
            _ = model(input_data)
    torch.cuda.synchronize()
    
    # Measure latency
    timings = []
    with torch.no_grad():
        for _ in range(nruns):
            start_time = time.time()
            _ = model(input_data)
            torch.cuda.synchronize()  # Ensure all ops are completed
            end_time = time.time()
            timings.append(end_time - start_time)
    
    avg_latency = sum(timings) / nruns
    print(f"Average Latency: {avg_latency * 1000:.2f} ms")
    return avg_latency


def calculate_model_size(model, dtype='fp32'):
    num_params = sum(p.numel() for p in model.parameters())
    
    if dtype == 'fp32':
        param_size = 4  # FP32 = 4 bytes
    elif dtype == 'fp16':
        param_size = 2  # FP16 = 2 bytes
    elif dtype == 'int8':
        param_size = 1  # INT8 = 1 byte
    elif dtype == 'int4':
        param_size = 0.5  # INT4 = 0.5 bytes
    else:
        raise ValueError("Unsupported data type")
    
    model_size = num_params * param_size / (1024 ** 2)  # Convert to MB
    print(f"Model Size: {model_size:.2f} MB")
    return model_size


In [ ]:
throughput = calculate_throughput(model, input_shape=(1, 3, 512, 1024), nwarmup=50, nruns=100)
latency = calculate_latency(model, input_shape=(1, 3, 512, 1024), nwarmup=50, nruns=100)
model_size = calculate_model_size(model, dtype='fp32')  # Example for FP16


In [ ]:
throughput = calculate_throughput(model, input_shape=(1, 3, 512, 1024), nwarmup=50, nruns=100)
latency = calculate_latency(model, input_shape=(1, 3, 512, 1024), nwarmup=50, nruns=100)
model_size = calculate_model_size(model, dtype='int8')  # Example for FP16


In [ ]:
def allocate_buffers(engine, max_batch_size, height, width, n_classes):
    inputs, outputs, bindings = [], [], []
    stream = cuda.Stream()
    for binding in range(engine.num_bindings):
        binding_shape = (
            (max_batch_size, 3, height, width) 
            if engine.binding_is_input(binding) 
            else (max_batch_size, n_classes, height, width)
        )
        size = trt.volume(binding_shape)
        dtype = trt.nptype(engine.get_binding_dtype(binding))
        host_mem = cuda.pagelocked_empty(size, dtype)
        device_mem = cuda.mem_alloc(host_mem.nbytes)
        bindings.append(int(device_mem))
        if engine.binding_is_input(binding):
            inputs.append({'host': host_mem, 'device': device_mem})
        else:
            outputs.append({'host': host_mem, 'device': device_mem})
    return inputs, outputs, bindings, stream

def infer_and_evaluate(engine, context, val_loader, criterion, device, n_classes, max_batch_size, height, width):
    # Allocate buffers once outside the loop
    inputs, outputs, bindings, stream = allocate_buffers(engine, max_batch_size, height, width, n_classes)
    total_loss, total_accuracy, total_iou, latencies = 0.0, 0.0, 0.0, []
    total_images = 0

    with torch.no_grad():
        for images, masks in val_loader:
            batch_size, channels, h, w = images.shape
            images, masks = images.to(device), masks.to(device).long()

            # Set dynamic shape for context
            context.set_binding_shape(0, (batch_size, channels, h, w))

            # Copy images to input buffer
            images = images.cpu().numpy().astype(np.float32)
            np.copyto(inputs[0]['host'], images.ravel())

            # Measure latency
            start_time = time.time()
            cuda.memcpy_htod_async(inputs[0]['device'], inputs[0]['host'], stream)
            context.execute_async_v2(bindings=bindings, stream_handle=stream.handle)
            cuda.memcpy_dtoh_async(outputs[0]['host'], outputs[0]['device'], stream)
            stream.synchronize()
            latency = time.time() - start_time
            latencies.append(latency)
            total_images += batch_size

            # Process model output
            output_shape = (batch_size, n_classes, h, w)  # Adjust based on context output
            main_output = torch.from_numpy(np.array(outputs[0]['host'])).view(*output_shape).to(device)
            loss = criterion(main_output, masks)
            accuracy = calculate_accuracy(main_output, masks)
            _, preds = torch.max(main_output, 1)
            iou = calculate_mean_iou(preds, masks, n_classes)

            total_loss += loss.item()
            total_accuracy += accuracy.item()
            total_iou += iou

    # Calculate average metrics
    avg_loss = total_loss / len(val_loader)
    avg_accuracy = total_accuracy / len(val_loader)
    avg_iou = total_iou / len(val_loader)
    avg_latency = np.mean(latencies)
    throughput = total_images / sum(latencies)

    return avg_loss, avg_accuracy, avg_iou, avg_latency, throughput


In [ ]:
from PIL import Image
import numpy as np
import torch
from torch.utils.data import DataLoader
import time
import pycuda.driver as cuda
import tensorrt as trt

# Define normalization parameters
mean = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1).cuda()  # For PyTorch
std = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1).cuda()  # For PyTorch

def allocate_buffers(engine, batch_size, height, width, n_classes):
    """
    Allocate buffers for TensorRT inference.
    """
    inputs, outputs, bindings = [], [], []
    stream = cuda.Stream()
    for binding in range(engine.num_bindings):
        binding_shape = (
            (batch_size, 3, height, width) 
            if engine.binding_is_input(binding) 
            else (batch_size, n_classes, height, width)
        )
        size = trt.volume(binding_shape)
        dtype = trt.nptype(engine.get_binding_dtype(binding))
        host_mem = cuda.pagelocked_empty(size, dtype)
        device_mem = cuda.mem_alloc(host_mem.nbytes)
        bindings.append(int(device_mem))
        if engine.binding_is_input(binding):
            inputs.append({'host': host_mem, 'device': device_mem})
        else:
            outputs.append({'host': host_mem, 'device': device_mem})
    return inputs, outputs, bindings, stream

def infer_and_evaluate(engine, context, val_loader, criterion, device, n_classes):
    """
    Run inference and evaluate the model using validation data.
    Assumes batch size = 1 and image shape = 512 x 1024.
    """
    # Fixed parameters for batch size and image dimensions
    batch_size = 16
    height, width = 512, 1024

    # Allocate buffers once
    inputs, outputs, bindings, stream = allocate_buffers(engine, batch_size, height, width, n_classes)

    total_loss, total_accuracy, total_iou, latencies = 0.0, 0.0, 0.0, []
    total_images = 0

    with torch.no_grad():
        for images, masks in val_loader:
            # Ensure batch size = 1
            images, masks = images[:batch_size].to(device), masks[:batch_size].to(device).long()

            # Normalize images directly on GPU
            images = (images - mean) / std  # Normalization on GPU

            # Copy images to input buffer
            np.copyto(inputs[0]['host'], images.cpu().numpy().ravel())

            # Measure latency
            start_time = time.time()
            cuda.memcpy_htod_async(inputs[0]['device'], inputs[0]['host'], stream)
            context.execute_async_v2(bindings=bindings, stream_handle=stream.handle)
            cuda.memcpy_dtoh_async(outputs[0]['host'], outputs[0]['device'], stream)
            stream.synchronize()
            latency = time.time() - start_time
            latencies.append(latency)
            total_images += batch_size

           

    # Calculate average metrics
    avg_latency = np.mean(latencies)
    throughput = total_images / sum(latencies)  # This will equal FPS for batch_size = 1

    return avg_latency, throughput


In [ ]:
import numpy as np
import torch
import time
import tensorrt as trt
import pycuda.driver as cuda
import pycuda.autoinit

def mean_iou(preds, targets, num_classes):
    """
    Compute Mean Intersection over Union (IoU) score.
    Args:
        preds (torch.Tensor): Model predictions (N, H, W) after argmax.
        targets (torch.Tensor): Ground truth masks (N, H, W).
        num_classes (int): Number of segmentation classes.
    Returns:
        float: mean IoU score.
    """
    iou_per_class = []
    for cls in range(num_classes):
        pred_inds = preds == cls
        target_inds = targets == cls

        intersection = (pred_inds & target_inds).sum().item()
        union = (pred_inds | target_inds).sum().item()

        if union == 0:
            iou_per_class.append(float('nan'))  # Ignore missing classes
        else:
            iou_per_class.append(intersection / union)

    return np.nanmean(iou_per_class)  # Compute mean IoU over valid classes

def infer_and_evaluate(engine, context, val_loader, criterion, device, num_classes, batch_sizes=[1]):
    """
    Run inference and evaluate the model using validation data.
    Computes accuracy, mean IoU, latency, and throughput.
    
    Args:
        engine (trt.ICudaEngine): TensorRT engine.
        context (trt.IExecutionContext): TensorRT execution context.
        val_loader (DataLoader): DataLoader for validation data.
        criterion (Loss Function): Loss function (e.g., CrossEntropyLoss).
        device (torch.device): Target device (CUDA).
        num_classes (int): Number of segmentation classes.
        batch_sizes (list): List of batch sizes to evaluate.
    
    Returns:
        dict: Accuracy, IoU, latency, and throughput for each batch size.
    """
    # Prepare evaluation metrics
    metrics = {
        batch_size: {"total_correct": 0, "total_samples": 0, "total_iou": 0, "latencies": []}
        for batch_size in batch_sizes
    }
    throughputs = {}

    mean = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1).to(device)
    std = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1).to(device)

    # Evaluate for each batch size
    for batch_size in batch_sizes:
        print(f"\nEvaluating for batch size: {batch_size}")

        # Set batch shape in context
        input_shape = (batch_size, 3, 512, 1024)
        context.set_binding_shape(0, input_shape)

        # Allocate GPU memory
        input_size = trt.volume(input_shape) * trt.float32.itemsize
        output_shape = context.get_binding_shape(1)
        output_size = trt.volume(output_shape) * trt.float32.itemsize

        d_input = cuda.mem_alloc(input_size)
        d_output = cuda.mem_alloc(output_size)
        h_output = np.empty(output_shape, dtype=np.float32)  # Host output buffer

        # Initialize timing and counters
        total_time = 0.0
        total_batches = 0

        with torch.no_grad():
            for images, masks in val_loader:
                if images.size(0) < batch_size:
                    break  # Ensure consistent batch sizes

                images, masks = images[:batch_size].to(device), masks[:batch_size].to(device)

                # Ensure correct normalization is applied
                images = (images - mean) / std  # Normalize on GPU

                # Transfer data to GPU
                cuda.memcpy_htod(d_input, images.cpu().numpy().astype(np.float32))

                # Measure latency
                start_time = time.time()
                context.execute_v2(bindings=[int(d_input), int(d_output)])
                cuda.memcpy_dtoh(h_output, d_output)
                end_time = time.time()

                latency = end_time - start_time
                metrics[batch_size]["latencies"].append(latency)
                total_time += latency
                total_batches += 1

                # Convert output to tensor
                output = torch.from_numpy(h_output).cuda().float()  # Ensure FP32 format
                output = torch.nn.functional.softmax(output, dim=1)  # Restore probability distribution
                preds = output.argmax(dim=1)  # Convert logits to class labels

                # Compute accuracy
                correct_pixels = (preds == masks).sum().item()
                total_pixels = masks.numel()
                metrics[batch_size]["total_correct"] += correct_pixels
                metrics[batch_size]["total_samples"] += total_pixels

                # Compute mean IoU
                iou = mean_iou(preds, masks, num_classes)
                metrics[batch_size]["total_iou"] += iou

        # Compute final metrics
        avg_latency = np.mean(metrics[batch_size]["latencies"]) * 1000  # Convert to ms
        throughput = (total_batches * batch_size) / total_time  # samples per second
        throughputs[batch_size] = throughput

        accuracy = metrics[batch_size]["total_correct"] / metrics[batch_size]["total_samples"]
        avg_iou = metrics[batch_size]["total_iou"] / total_batches

        # Print results
        print(f"Batch Size: {batch_size}")
        print(f"Accuracy: {accuracy * 100:.2f}%")
        print(f"Mean IoU: {avg_iou:.4f}")
        print(f"Average Latency: {avg_latency:.2f} ms")
        print(f"Throughput: {throughput:.2f} samples/second")

    return metrics, throughputs


In [ ]:
import tensorrt as trt
import pycuda.driver as cuda
import pycuda.autoinit  # Automatically initializes CUDA driver


# Define necessary parameters
engine_path = "/kaggle/working/mobilenetv3_base.trt"
device = torch.device("cuda")  # or torch.device("cpu")
n_classes = 19  # Set according to your dataset, e.g., 19 for Cityscapes

# Load the TensorRT engine
TRT_LOGGER = trt.Logger(trt.Logger.WARNING)

def load_engine(engine_path):
    """
    Load a serialized TensorRT engine from file.
    """
    with open(engine_path, "rb") as f, trt.Runtime(TRT_LOGGER) as runtime:
        engine = runtime.deserialize_cuda_engine(f.read())
        if engine is None:
            raise RuntimeError(f"Failed to load TensorRT engine from {engine_path}")
        return engine

engine = load_engine(engine_path)

# Create an execution context from the engine
context = engine.create_execution_context()

# Ensure context supports dynamic input shapes (if applicable)
if not context:
    raise RuntimeError("Failed to create execution context from TensorRT engine.")

# Call the function
avg_latency, throughput = infer_and_evaluate(engine, context, val_loader, criterion, device, n_classes)


In [ ]:
#FP32
Evaluating for batch size: 16
Batch Size: 16
Accuracy: 7.20%
Mean IoU: 0.0374
Average Latency: 324.57 ms
Throughput: 49.30 samples/second

#FP16
Evaluating for batch size: 16
Batch Size: 16
Accuracy: 7.19%
Mean IoU: 0.0374
Average Latency: 238.58 ms
Throughput: 67.06 samples/second

#INT8

Evaluating for batch size: 16
Batch Size: 16
Accuracy: 51.95%
Mean IoU: 0.0987
Average Latency: 226.94 ms
Throughput: 70.50 samples/second

